# 02 · Faster R-CNN (two-stage, anchor-based)

Rubric: *Two-stage Anchor-based*. Same protocol as every other model: 3-fold GroupKFold by clip
(`splits.json`), train frames every 20th frame, validation frames every 10th frame, 640 px,
equal epochs, OOF COCO metrics with ignored regions as `iscrowd=1` (`common.evaluate`).

Training and evaluation code lives in `tv_train.py` (shared with RetinaNet); data loading in
`tv_dataset.py`; metrics and logging in `common.py`.

How to run: set `MODE`, `EXPERIMENT` and `FOLDS` below. `quick` checks every code path in a few
minutes; `full` trains the experiment on all folds (one experiment per Kaggle commit).

## 1. Run configuration
`MODE` is `"quick"` or `"full"`; `EXPERIMENT` is a key of `EXPERIMENTS` (section 3); `FOLDS` are the folds
trained in this session. `RUN_ANALYSIS` enables the failure-case / RPN analysis (always on in quick mode).

In [ ]:
MODE = "quick"
EXPERIMENT = "frcnn_base"
FOLDS = [0, 1, 2]
RUN_ANALYSIS = False
ANALYSIS_EXPERIMENT = "auto"
PROPOSAL_SWEEP = [300, 1000, 2000]

## 2. Setup
Use the shared modules next to this notebook (repo) or download them from the GitHub repo (Kaggle).

In [ ]:
import os
import sys
import time
import urllib.request
from pathlib import Path

REPO_RAW = "https://raw.githubusercontent.com/KaplyshOlha/hw-2/main/notebooks/"
SRC_FILES = ["common.py", "tv_dataset.py", "tv_train.py", "splits.json", "anchors.json"]
SRC = next((Path(p) for p in [".", "notebooks"] if all((Path(p) / f).exists() for f in SRC_FILES)), None)
if SRC is None:
    SRC = Path("/tmp/hw2_src")
    SRC.mkdir(parents=True, exist_ok=True)
    for f in SRC_FILES:
        urllib.request.urlretrieve(REPO_RAW + f, SRC / f)
sys.path.insert(0, str(SRC.resolve()))

Imports, output paths and device. FULL mode stops immediately if no GPU is attached.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

import common
import tv_train as tvt

assert MODE in ("quick", "full")
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
RUN_ROOT = WORK / ("runs" if MODE == "full" else "runs_quick")
RESULTS = WORK / ("results.csv" if MODE == "full" else "results_quick.csv")
IMG_DIR = WORK / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = os.environ.get("HW2_INPUTS", "/kaggle/input")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if MODE == "full":
    assert DEVICE.type == "cuda", "FULL mode needs a GPU (Settings -> Accelerator -> GPU T4)"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 130, "savefig.bbox": "tight"})
print(f"MODE={MODE} EXPERIMENT={EXPERIMENT} FOLDS={FOLDS} "
      f"device={torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'} src={SRC}")

Copy finished or interrupted folds (and `results.csv`) from previous notebook versions attached
as input, so a new session continues instead of retraining.

In [ ]:
tvt.restore_from_inputs(RUN_ROOT, RESULTS, INPUT_ROOT, name=RUN_ROOT.name);

## 3. Experiments
All runs share the protocol in `tvt.Config` (SGD, momentum 0.9, wd 1e-4, batch 8, LR 0.01 = 0.02·8/16,
200 warmup iterations, 12 epochs with ×0.1 at epochs 8 and 11, AMP, horizontal flip, ignored regions
painted gray in training frames, raw validation frames like the test set).

| experiment | change | motivation |
|---|---|---|
| `frcnn_base` | torchvision defaults, new 9-class head, 640 px, `score_thresh=0.001`, `detections_per_img=300` | reference; EDA: up to 180 objects per frame |
| `frcnn_anch05` | anchors ×0.5 → 16, 32, 64, 128, 256 (one per FPN level) | EDA: median object 15 px at 640; a 15×15 box has IoU 0.22 with the smallest default 32×32 anchor |
| `frcnn_roi05` | `frcnn_anch05` + RoI positive fraction 0.25 → 0.5 | 512 RoIs × 0.25 = at most 128 positives, but 7.5% of frames have >100 objects |
| `frcnn_giou` | `frcnn_anch05` + GIoU box loss on decoded boxes (weight 10) | smooth L1 on deltas ignores box overlap; IoU-type loss is scale-invariant ([GIoU](https://arxiv.org/abs/1902.09630), weight from mmdetection) |
| `frcnn_roi05_giou` | both changes (run only if both help) | final two-stage model |

`frcnn_roi05` and `frcnn_giou` each change one thing relative to `frcnn_anch05`, so their effect can be
attributed. The number of test-time RPN proposals is studied without retraining in section 7.

In [ ]:
BASE = tvt.Config(experiment="frcnn_base", model="faster_rcnn", anchors_json=str(SRC / "anchors.json"),
                  notes="torchvision defaults, COCO-pretrained, 640px")
EXPERIMENTS = {"frcnn_base": BASE}
EXPERIMENTS["frcnn_anch05"] = tvt.make_config(
    BASE, experiment="frcnn_anch05", anchor_key="x0.5", notes="anchors x0.5 (16-256)")
EXPERIMENTS["frcnn_roi05"] = tvt.make_config(
    EXPERIMENTS["frcnn_anch05"], experiment="frcnn_roi05", box_positive_fraction=0.5,
    notes="anch05 + RoI positive fraction 0.5")
EXPERIMENTS["frcnn_giou"] = tvt.make_config(
    EXPERIMENTS["frcnn_anch05"], experiment="frcnn_giou", box_loss="giou", box_loss_weight=10.0,
    notes="anch05 + GIoU box loss (w=10)")
EXPERIMENTS["frcnn_roi05_giou"] = tvt.make_config(
    EXPERIMENTS["frcnn_roi05"], experiment="frcnn_roi05_giou", box_loss="giou", box_loss_weight=10.0,
    notes="anch05 + RoI pos 0.5 + GIoU")

QUICK = dict(epochs=2, max_iters=15, batch_size=2, train_stride=100, eval_stride=200,
             warmup_iters=10, milestones=(1,), fps_warmup=3, fps_iters=10)


def get_cfg(name):
    c = EXPERIMENTS[name]
    return tvt.make_config(c, **QUICK) if MODE == "quick" else c


cfg = get_cfg(EXPERIMENT)
pd.Series({k: v for k, v in cfg.key().items() if v != tvt.Config().key().get(k)}, name="changed vs defaults", dtype=object)

## 4. Data and folds
Load all labelled frames (train + val) and the fixed 3-fold split by clip; show how many frames each
fold trains and validates on with the strides of this run.

In [ ]:
common.seed_everything()
images, anns = common.load_gt(cache="/tmp/gt_cache.pkl")
splits = common.load_splits(SRC / "splits.json")
pd.DataFrame([{"fold": f["fold"], "train_clips": len(f["train_clips"]), "val_clips": len(f["val_clips"]),
               "train_frames": len(common.fold_images(images, splits, f["fold"], "train", cfg.train_stride)),
               "val_frames": len(common.fold_images(images, splits, f["fold"], "val", cfg.eval_stride))}
              for f in splits["folds"]]).set_index("fold")

## 5. Train and predict
Train every fold in `FOLDS` (finished folds are skipped, interrupted ones resume from `last.pt`), predict
the fold's validation frames and measure FPS. In QUICK mode an interruption after epoch 1 is simulated
first, so the resume path is tested too.

In [ ]:
t0 = time.time()
if MODE == "quick":
    try:
        tvt.run_experiment(tvt.make_config(cfg, stop_after_epoch=1), images, anns, splits, FOLDS[:1],
                           RUN_ROOT, RESULTS, DEVICE)
    except tvt.Interrupted as e:
        print("resume test:", e)
metrics = tvt.run_experiment(cfg, images, anns, splits, FOLDS, RUN_ROOT, RESULTS, DEVICE,
                             require_all_folds=(MODE == "full"))
print(f"total {(time.time() - t0) / 60:.1f} min")

OOF metrics of this experiment: per fold, mean ± std over folds, and pooled out-of-fold (`oof`).

In [ ]:
MAIN = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500"]
PER_CLASS = [f"AP_{c}" for c in common.CLASSES]
if metrics is not None:
    display(metrics[MAIN].round(4))
    display(metrics.loc[["oof"], PER_CLASS].round(4))

## 6. Comparison of experiments
Latest logged run of every Faster R-CNN experiment in `results.csv` (OOF and mean ± std over folds).

In [ ]:
names = [n for n in EXPERIMENTS if (RUN_ROOT / n).exists()]
summ = tvt.summary(RESULTS, model="faster_rcnn", experiments=names)
if len(summ):
    table = pd.DataFrame({m: summ[f"{m}_oof"].round(4) for m in ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AR500"]})
    table["mAP mean±std"] = [f"{a:.4f} ± {b:.4f}" for a, b in zip(summ["mAP_mean"], summ["mAP_std"])]
    table["FPS"] = summ["fps"].round(1)
    table["train min/fold"] = summ["train_min"].round(1)
    table.to_csv(WORK / "02_frcnn_comparison.csv")
    display(table)

Main metrics with fold std as error bars, and per-class OOF AP of each experiment.

In [ ]:
if len(summ):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 1.3]})
    ms = ["mAP", "AP50", "AP_small", "AP_medium"]
    x = np.arange(len(ms))
    w = 0.8 / len(summ)
    for i, (n, r) in enumerate(summ.iterrows()):
        ax[0].bar(x + i * w, [r[f"{m}_mean"] for m in ms], w, yerr=[r[f"{m}_std"] for m in ms],
                  capsize=3, label=n)
    ax[0].set_xticks(x + w * (len(summ) - 1) / 2, ms)
    ax[0].set_title("Faster R-CNN: mean over 3 folds (± std)")
    ax[0].legend(fontsize=8)
    pc = summ[[c for c in PER_CLASS if c in summ]].astype(float)
    im = ax[1].imshow(pc.values, cmap="viridis", aspect="auto")
    ax[1].set_xticks(range(pc.shape[1]), [c[3:] for c in pc.columns], rotation=30)
    ax[1].set_yticks(range(len(pc)), pc.index)
    for (i, j), v in np.ndenumerate(pc.values):
        ax[1].text(j, i, f"{v:.2f}", ha="center", va="center", color="w", fontsize=8)
    ax[1].set_title("OOF AP per class")
    fig.colorbar(im, ax=ax[1])
    fig.savefig(IMG_DIR / "02_frcnn_comparison.png")
    plt.show()

Training loss per epoch (fold 0) for every finished experiment.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for n in names:
    done = tvt.load_done(RUN_ROOT, n)
    if 0 not in done:
        continue
    h = pd.DataFrame(done[0]["history"])
    for a, col in zip(ax, ["loss", "loss_box_reg", "loss_objectness"]):
        if col in h:
            a.plot(h["epoch"], h[col], marker="o", ms=3, label=n)
            a.set_title(col)
            a.set_xlabel("epoch")
ax[0].legend(fontsize=8)
fig.savefig(IMG_DIR / "02_frcnn_loss_curves.png")
plt.show()

**Observations (comparison):** TBD

## 7. Analysis setup
Pick the experiment to analyse (`auto` = best OOF mAP among finished experiments). The analysis needs
the saved `final.pt` weights of its folds.

In [ ]:
DO_ANALYSIS = RUN_ANALYSIS or MODE == "quick"
if ANALYSIS_EXPERIMENT == "auto":
    ANALYSIS_EXPERIMENT = summ["mAP_oof"].idxmax() if len(summ) else EXPERIMENT
acfg = get_cfg(ANALYSIS_EXPERIMENT)
adone = sorted(tvt.load_done(RUN_ROOT, ANALYSIS_EXPERIMENT))
apreds = tvt.load_fold_preds(RUN_ROOT, ANALYSIS_EXPERIMENT)
DO_ANALYSIS = DO_ANALYSIS and len(adone) > 0
print(f"analysis of {ANALYSIS_EXPERIMENT}, folds {adone}, enabled={DO_ANALYSIS}")

### 7.1 Number of RPN proposals at test time
The same trained weights are evaluated with 300 / 1000 / 2000 proposals per image (pre-NMS top-n per
level = max(1000, N)). This isolates the proposal budget from training effects; each setting is logged
to `results.csv` as `<experiment>_propN` with `notes="inference-only"`.

In [ ]:
sweep = {}
if DO_ANALYSIS:
    for n in PROPOSAL_SWEEP:
        if n == acfg.rpn_post_nms_top_n_test:
            p, fps = apreds, {k: d["fps"] for k, d in tvt.load_done(RUN_ROOT, ANALYSIS_EXPERIMENT).items()}
        else:
            p, fps = tvt.predict_from_checkpoints(acfg, images, anns, splits, RUN_ROOT, DEVICE,
                                                  rpn_topn=(max(1000, n), n))
        m = common.evaluate_oof(p, images, anns, splits, eval_stride=acfg.eval_stride)
        sweep[n] = {**m.loc["oof", ["mAP", "AP50", "AP_small", "AR500"]].to_dict(),
                    "mAP_std": m.loc["std", "mAP"], "fps": float(np.nanmean(list(fps.values())))}
        logged = set(pd.read_csv(RESULTS)["experiment"]) if RESULTS.exists() else set()
        name = f"{ANALYSIS_EXPERIMENT}_prop{n}"
        if (n != acfg.rpn_post_nms_top_n_test and MODE == "full" and len(p) == splits["n_folds"]
                and name not in logged):
            common.log_results(m, path=RESULTS, owner=acfg.owner, model=acfg.model,
                               experiment=name, train_stride=acfg.train_stride,
                               eval_stride=acfg.eval_stride, imgsz=acfg.imgsz, epochs=acfg.epochs,
                               fps=round(sweep[n]["fps"], 1), notes=f"inference-only: {n} RPN proposals")
    sweep = pd.DataFrame(sweep).T.rename_axis("proposals")
    display(sweep.round(4))

OOF mAP / AR500 and speed as a function of the proposal count.

In [ ]:
if len(sweep):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    for col in ["mAP", "AP_small", "AR500"]:
        ax[0].plot(sweep.index, sweep[col], marker="o", label=col)
    ax[0].set_xscale("log")
    ax[0].set_xlabel("RPN proposals per image (test)")
    ax[0].legend()
    ax[1].plot(sweep.index, sweep["fps"], marker="o", color="k")
    ax[1].set_xscale("log")
    ax[1].set_title("FPS (batch 1)")
    fig.savefig(IMG_DIR / "02_frcnn_proposal_sweep.png")
    plt.show()

**Observations (proposal count):** TBD

### 7.2 What the RPN and the second stage contribute
**RPN (stage 1).** A small conv head slides over every FPN level and, for each anchor, predicts an
objectness score and a box offset. After NMS it keeps the top-N boxes as class-agnostic *proposals*.
Its job is recall: an object that no proposal covers cannot be detected later. It also removes the
huge background imbalance, because the second stage sees ~1000 candidates instead of ~10⁵ anchors.

**RoI head (stage 2).** RoIAlign crops a fixed 7×7 feature for each proposal from the FPN level that
matches its size; two FC layers then classify it into 8 classes + background and regress a second,
class-specific box correction. Its job is precision: rejecting background proposals, choosing the
class and refining the box.

Below, for frames of fold 0, we measure recall of the top-N proposals (any IoU ≥ 0.5 with a GT box,
class-agnostic) against recall of the final detections (score ≥ 0.05; any class vs correct class),
by GT size at network scale (640 px), and the mean best IoU before and after the second stage.

In [ ]:
if DO_ANALYSIS:
    amodel = tvt.load_model(acfg, adone[0], RUN_ROOT, DEVICE)
    _, ava = tvt.make_datasets(acfg, images, anns, splits, adone[0], precache=False)
    pr = tvt.proposal_recall(amodel, ava, DEVICE, topn=(100, 300, 1000, 2000), max_images=300)
    cols = [c for c in pr if c.startswith("prop@")] + ["det_any_class", "det_correct_class"]
    rpn_tab = pr.groupby("size_bin", observed=True)[cols + ["prop_best_iou", "det_best_iou"]].mean()
    rpn_tab.insert(0, "n_gt", pr.groupby("size_bin", observed=True).size())
    rpn_tab.loc["all"] = [len(pr)] + pr[cols + ["prop_best_iou", "det_best_iou"]].mean().tolist()
    display(rpn_tab.round(3))
    del amodel

Recall by object size: RPN proposals (top-N) vs final detections.

In [ ]:
if DO_ANALYSIS:
    t = rpn_tab.drop(index="all")
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for c in cols:
        ax.plot(t.index.astype(str), t[c], marker="o", ls="--" if c.startswith("prop") else "-", label=c)
    ax.set_xlabel("GT size sqrt(w·h) at 640 px")
    ax.set_ylabel("recall @ IoU 0.5")
    ax.legend(fontsize=8)
    fig.savefig(IMG_DIR / "02_frcnn_rpn_vs_head.png")
    plt.show()

**Observations (RPN vs second stage):** TBD

### 7.3 Error analysis on OOF predictions
Detections with score ≥ 0.3 are matched greedily to GT (IoU ≥ 0.5, same class). False positives are
typed as `cls` (overlaps a GT of another class), `loc` (same class, 0.1 ≤ IoU < 0.5), `dup` (GT already
matched) or `bkg`; detections mostly inside ignored regions are dropped. Sizes are in original pixels.

In [ ]:
if DO_ANALYSIS:
    oof_ids = np.concatenate([common.fold_images(images, splits, k, "val", acfg.eval_stride)["img_id"].to_numpy()
                              for k in sorted(apreds)])
    gt, dt = tvt.error_analysis(pd.concat(apreds.values(), ignore_index=True), images, anns, oof_ids,
                                score_thr=0.3)
    gt["density"] = pd.cut(gt["n_in_frame"], [0, 20, 50, 100, 1000], labels=["≤20", "21-50", "51-100", ">100"])
    summary_err = pd.Series({"GT objects": len(gt), "recall": gt["matched"].mean(),
                             **dt["type"].value_counts(normalize=True).rename(lambda s: f"dets {s}").to_dict()})
    display(summary_err.round(3).to_frame("value").T)

Recall by size, occlusion and scene density, and which class a confused GT object was predicted as.

In [ ]:
if DO_ANALYSIS:
    fig, ax = plt.subplots(1, 4, figsize=(18, 3.8), gridspec_kw={"width_ratios": [1, 0.6, 0.8, 1.2]})
    for a, col, title in zip(ax[:3], ["size_bin", "occlusion", "density"],
                             ["GT size (orig px)", "occlusion (0 none, 1 part, 2 heavy)", "objects per frame"]):
        r = gt.groupby(col, observed=True)["matched"].agg(["mean", "size"])
        a.bar(r.index.astype(str), r["mean"])
        for i, (v, s) in enumerate(zip(r["mean"], r["size"])):
            a.text(i, v + 0.01, f"n={s}", ha="center", fontsize=7)
        a.set_title(f"recall vs {title}", fontsize=9)
        a.set_ylim(0, 1)
    conf = gt[gt["confused_with"] >= 0]
    cm = pd.crosstab(conf["cls"], conf["confused_with"]).reindex(index=range(8), columns=range(8), fill_value=0)
    ax[3].imshow(cm.values, cmap="Reds")
    for (i, j), v in np.ndenumerate(cm.values):
        if v:
            ax[3].text(j, i, v, ha="center", va="center", fontsize=7)
    ax[3].set_xticks(range(8), common.CLASSES, rotation=45, fontsize=7)
    ax[3].set_yticks(range(8), common.CLASSES, fontsize=7)
    ax[3].set_xlabel("predicted as")
    ax[3].set_ylabel("GT class")
    ax[3].set_title("missed GT confused with another class", fontsize=9)
    fig.savefig(IMG_DIR / "02_frcnn_error_breakdown.png")
    plt.show()

### 7.4 Failure cases
Four frames chosen automatically: most missed small objects (< 16 px), most class confusions,
densest frame, most background false positives. Colours: green TP, red missed GT, magenta wrong class,
orange poor localisation, blue duplicate, yellow background FP; gray = ignored region. Crops are
centred on the errors.

In [ ]:
if DO_ANALYSIS:
    per = gt.assign(small_miss=(~gt["matched"]) & (gt["side"] < 16), conf=gt["confused_with"] >= 0)
    per = per.groupby("img_id").agg(small_miss=("small_miss", "sum"), conf=("conf", "sum"),
                                    n=("cls", "size"), recall=("matched", "mean"))
    per["bkg"] = dt[dt["type"] == "bkg"].groupby("img_id").size().reindex(per.index, fill_value=0)
    cases = {"most missed small objects": per["small_miss"].idxmax(),
             "most class confusions": per["conf"].idxmax(),
             "densest frame": per["n"].idxmax(),
             "most background FPs": per["bkg"].idxmax()}
    fig, ax = plt.subplots(2, 2, figsize=(16, 10))
    for a, (title, iid) in zip(ax.ravel(), cases.items()):
        a.imshow(tvt.draw_errors(iid, images, anns, gt, dt, crop="auto"))
        r = per.loc[iid]
        a.set_title(f"{title}: {images.loc[iid, 'clip']} #{images.loc[iid, 'frame_id']} "
                    f"(n={r['n']}, recall={r['recall']:.2f})", fontsize=9)
        a.axis("off")
    fig.savefig(IMG_DIR / "02_frcnn_failure_cases.png")
    plt.show()

**Failure case 1 — tiny objects:** TBD

**Failure case 2 — class confusion:** TBD

**Failure case 3 — dense scenes:** TBD

**Failure case 4 — background false positives:** TBD

## 8. Outputs
In FULL mode, merge this session's `results.csv` with the current repo copy into `results_merged.csv`
(upload it to the repo as `notebooks/results.csv`), and list the files to download.

In [ ]:
if MODE == "full" and RESULTS.exists():
    merged = WORK / "results_merged.csv"
    try:
        urllib.request.urlretrieve(REPO_RAW + "results.csv", merged)
    except Exception as e:
        print("repo results.csv not available:", e)
        merged.unlink(missing_ok=True)
    common.merge_results(str(RESULTS), str(merged))
    print(f"{merged}: {len(pd.read_csv(merged))} rows")
print("images:", sorted(p.name for p in IMG_DIR.glob("02_*.png")))

## 9. Conclusions
TBD